## Импорты

In [1]:
try:
    import google.colab
    %pip install pandas torch transformers datasets evaluate sacrebleu bert-score sentence-transformers rapidfuzz deep-translator tqdm numpy matplotlib PyYAML
    %pip install --upgrade tiktoken protobuf sentencepiece
    %pip install accelerate>=1.1.0
    is_colab = True
except:
    is_colab = False

if is_colab:
    !git clone https://github.com/AbonentVneSeti/text_processing_final_project
    %cd text_processing_final_project

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 78.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 32.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: tiktoken
    Found existing installation: tiktoken 0.12.0
    Uninstalling tiktoken-0.12.0:
      Successfully uninstalled tiktoken-0.12.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This b

Cloning into 'text_processing_final_project'...
remote: Enumerating objects: 95, done.
remote: Counting objects: 100% (95/95), done.
remote: Compressing objects: 100% (67/67), done.
remote: Total 95 (delta 46), reused 72 (delta 23), pack-reused 0 (from 0)
Receiving objects: 100% (95/95), 11.10 MiB | 17.85 MiB/s, done.
Resolving deltas: 100% (46/46), done.
/content/text_processing_final_project


In [2]:
import sys
sys.path.append('..')
from utils import load_config, build_dataloaders
import pandas as pd
from models.trainer import train_model
from models.rut5_paraphraser.model import ParaphraserModel

config = load_config("config.yaml")

## Загрузка датасета

In [3]:
import os
if not os.path.exists("data/paraphrases_clean.parquet"):
    !wget -O data/paraphrases_clean.parquet https://github.com/AbonentVneSeti/text_processing_final_project/main/data/paraphrases_clean.parquet

df = pd.read_parquet("data/paraphrases_clean.parquet")
print(f"Загружено {len(df)} пар")

Загружено 101945 пар


In [4]:
train_loader, val_loader = build_dataloaders(df, config["model_config"])

## Обучение модели

In [5]:
model = ParaphraserModel(config["model_config"])
trained_model = train_model(
    model, train_loader, val_loader,
    config["model_config"], config["trainer_config"], config["metrics_config"]
)
trained_model.save(config["trainer_config"]["output_dir"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/664 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/233 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/828k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Map:   0%|          | 0/91750 [00:00<?, ? examples/s]

Map:   0%|          | 0/10195 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

## Сохранение (для Colab)

In [ ]:
if is_colab:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r models/rut5_paraphraser/saves /content/drive/MyDrive/
    print("Модель сохранена на Google Drive")

In [8]:
if is_colab:
    google.colab.runtime.unassign()